In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 19 — Reference coding solutions

Three exercises. Read the lesson and workbook first. Learner functions intentionally raise `NotImplementedError` until you write them. Reference execution is not your assessment.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/shape_lab").is_dir())
if str(ROOT/"src") not in sys.path:sys.path.insert(0,str(ROOT/"src"))


## 19.C1 — `decision_delay`

Return causal decision delay, rejecting a decision before the estimated onset.

**Before coding:** work a small example on paper.

**Hint:** Decision time is when sufficient evidence becomes available, not necessarily the event onset.

In [3]:
def decision_delay(onset_time, decision_time):
    if not np.isfinite([onset_time,decision_time]).all() or decision_time<onset_time:raise ValueError("Decision must be finite and no earlier than onset.")
    return decision_time-onset_time

In [4]:
assert np.isclose(decision_delay(.24,.26),.02)
try:decision_delay(.3,.2)
except ValueError:pass
else:raise AssertionError("Negative decision delay is invalid under this contract.")
print("19.C1: checks passed")

19.C1: checks passed


**Why this works and where it stops:** A retrospectively assigned onset timestamp does not mean an online decision was available then.

## 19.C2 — `visible_pair_mask`

Allow a pair only when both observations are visible and the identity pairing is valid. Inputs are one-dimensional matching boolean arrays.

**Before coding:** work a small example on paper.

**Hint:** Use conjunction, not a union of evidence.

In [5]:
def visible_pair_mask(visible_a, visible_b, identity_matches):
    a=np.asarray(visible_a,bool);b=np.asarray(visible_b,bool);i=np.asarray(identity_matches,bool)
    if a.ndim!=1 or a.shape!=b.shape or a.shape!=i.shape:raise ValueError("Matching one-dimensional masks required.")
    return a&b&i

In [6]:
assert np.array_equal(visible_pair_mask([1,1,0],[1,0,1],[1,1,1]),[True,False,False])
assert not visible_pair_mask([1],[1],[0])[0]
print("19.C2: checks passed")

19.C2: checks passed


**Why this works and where it stops:** Valid visibility and identity are prerequisites, not sufficient evidence of physical contact. Predicted positions cannot be relabeled as direct observations.

## 19.C3 — `event_rates`

Return precision and recall from nonnegative integer counts. Undefined rates are None.

**Before coding:** work a small example on paper.

**Hint:** Precision denominator uses predicted events; recall denominator uses reference events.

In [7]:
def event_rates(true_positive, false_positive, false_negative):
    vals=[true_positive,false_positive,false_negative]
    if any(not isinstance(v,(int,np.integer)) or v<0 for v in vals):raise ValueError("Nonnegative integer counts required.")
    tp,fp,fn=vals
    return {"precision":tp/(tp+fp) if tp+fp else None,"recall":tp/(tp+fn) if tp+fn else None}

In [8]:
assert event_rates(2,1,2)=={'precision':2/3,'recall':.5}
assert event_rates(0,0,0)=={'precision':None,'recall':None}
print("19.C3: checks passed")

19.C3: checks passed


**Why this works and where it stops:** These counts must come from a declared one-to-one matching protocol. Correct arithmetic does not establish valid annotations or an appropriate event definition.